# 07. Urban Flood Hazard Index

This notebook combines the maximum depth and maximum velocity rasters of the 20 HEC-RAS simulations (four return periods under five climates) into the Urban Flood Hazard Index (UFHI) and classifies it into five hazard classes. It produces the entropy weights, the class breaks, the classified hazard rasters, the class-area tables and the summary of maximum depth, wetted area and high-hazard area.

To keep the 20 maps comparable, the normalisation bounds, the entropy weights and the Jenks class breaks are derived once from the pooled values of all 20 cases and applied unchanged to every map, so that each class covers the same index range in all cases.


## Configuration

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import jenkspy

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)

ROOT = Path("..")
RETURN_PERIODS = ["2yr", "10yr", "50yr", "100yr"]
SCENARIOS = ["Observed", "SSP245_2021_2050", "SSP245_2051_2080", "SSP585_2021_2050", "SSP585_2051_2080"]

DEPTH_PATTERN = str(ROOT / "data" / "hecras" / "depth_{rp}_{scen}.tif")
VELOCITY_PATTERN = str(ROOT / "data" / "hecras" / "velocity_{rp}_{scen}.tif")
WATERBODIES = ROOT / "data" / "waterbodies" / "waterbodies.shp"
OUT_DIR = ROOT / "results" / "07_ufhi"

NCLASS = 5
HAZ_LABELS = ["Very Low", "Low", "Medium", "High", "Very High"]
JENKS_SAMPLE = 150000          # values sampled from the pooled index for the Jenks breaks
JENKS_SEED = 0
NODATA = -9999.0

OUT_DIR.mkdir(parents=True, exist_ok=True)
CASES = [(rp, s) for rp in RETURN_PERIODS for s in SCENARIOS]

def dpath(rp, s): return DEPTH_PATTERN.format(rp=rp, scen=s)
def vpath(rp, s): return VELOCITY_PATTERN.format(rp=rp, scen=s)

missing = [p for rp, s in CASES for p in (dpath(rp, s), vpath(rp, s)) if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} input raster(s) not found, for example {missing[0]}")
print(f"{len(CASES)} cases = {len(RETURN_PERIODS)} return periods x {len(SCENARIOS)} climates; all rasters found")

## 1. Raster helpers

In [ ]:
def read(p):
    """Read band 1 as float64 with nodata set to NaN; return the array and the raster profile."""
    with rasterio.open(p) as s:
        a = s.read(1).astype("float64"); prof = s.profile
        if s.nodata is not None: a[a == s.nodata] = np.nan
    return a, prof

def wb_mask(prof):
    """True for cells inside the waterbody polygons."""
    g = gpd.read_file(WATERBODIES).to_crs(prof["crs"])
    return geometry_mask(g.geometry, out_shape=(prof["height"], prof["width"]),
                         transform=prof["transform"], invert=True)

def save(path, arr, prof, nodata=NODATA):
    p = prof.copy(); p.update(dtype="float32", count=1, nodata=nodata, compress="lzw")
    with rasterio.open(path, "w", **p) as d:
        d.write(np.where(np.isfinite(arr), arr, nodata).astype("float32"), 1)

def valid(a):
    return a[np.isfinite(a)]

## 2. Waterbody mask and momentum

Cells within the mapped ponds and khals are excluded from depth and velocity. The flow momentum per unit width is $M = D \times V$ 

In [ ]:
masked = {}
for rp, s in CASES:
    d, prof = read(dpath(rp, s)); v, _ = read(vpath(rp, s)); m = wb_mask(prof)
    d[m] = np.nan; v[m] = np.nan; mom = d * v
    tag = f"{rp}_{s}"
    save(OUT_DIR / f"depth_masked_{tag}.tif", d, prof)
    save(OUT_DIR / f"velocity_masked_{tag}.tif", v, prof)
    save(OUT_DIR / f"momentum_{tag}.tif", mom, prof)
    masked[(rp, s)] = dict(D=d, V=v, M=mom, prof=prof)
print(f"{len(CASES)} cases masked; momentum computed and saved.")

## 3. Pooled normalisation bounds and entropy weights

Each variable is rescaled by min-max normalisation with its pooled minimum and maximum over all 20 cases, $x_{\mathrm{norm}} = (x - x_{\min})/(x_{\max} - x_{\min})$. The weights follow the Shannon entropy method (Shannon 1948; Zou et al. 2006). For criterion $j$ the normalised values are converted to shares $p_{ij} = x_{ij}/\sum_i x_{ij}$ over the $m_c$ pooled valid cells, and

$$e_j = -\frac{1}{\ln m_c}\sum_{i=1}^{m_c} p_{ij}\ln p_{ij}, \qquad w_j = \frac{1 - e_j}{\sum_{l=1}^{3}\left(1 - e_l\right)}$$

with $p_{ij}\ln p_{ij} = 0$ when $p_{ij} = 0$. The weights reflect how unevenly each variable is distributed across the cells, not its physical severity.

In [ ]:
pool = {c: np.concatenate([valid(masked[k][c]) for k in CASES]) for c in ["D", "V", "M"]}
BOUNDS = {c: (float(pool[c].min()), float(pool[c].max())) for c in ["D", "V", "M"]}

def normalize(a, c):
    lo, hi = BOUNDS[c]
    return np.clip((a - lo) / (hi - lo), 0, None) if hi > lo else np.zeros_like(a)

cols = {c: normalize(pool[c], c) for c in ["D", "V", "M"]}
m = len(cols["D"]); k = 1 / np.log(m); diver = {}
for c, x in cols.items():
    ss = x.sum(); p = x / ss if ss > 0 else np.zeros_like(x)
    with np.errstate(divide="ignore", invalid="ignore"):
        plnp = np.where(p > 0, p * np.log(p), 0.0)
    diver[c] = 1 - (-k * plnp.sum())
tot = sum(diver.values()); WEIGHTS = {c: diver[c] / tot for c in diver}

NAMES = {"D": "Depth (D)", "V": "Velocity (V)", "M": "Momentum (M = D x V)"}
btab = pd.DataFrame({"Criterion": [NAMES[c] for c in "DVM"],
                     "Pooled minimum": [BOUNDS[c][0] for c in "DVM"],
                     "Pooled maximum": [BOUNDS[c][1] for c in "DVM"]})
btab.to_csv(OUT_DIR / "normalisation_bounds.csv", index=False)
wtab = pd.DataFrame({"Criterion": [NAMES[c] for c in "DVM"],
                     "Diversification": [round(diver[c], 4) for c in "DVM"],
                     "Entropy weight": [round(WEIGHTS[c], 4) for c in "DVM"]})
wtab.to_csv(OUT_DIR / "entropy_weights.csv", index=False)
print("Pooled bounds:", {c: (round(BOUNDS[c][0], 3), round(BOUNDS[c][1], 3)) for c in BOUNDS})
wtab

## 4. Urban Flood Hazard Index

$$\mathrm{UFHI} = w_D D_{\mathrm{norm}} + w_V V_{\mathrm{norm}} + w_M M_{\mathrm{norm}}$$

In [ ]:
ufhi = {}
for rp, s in CASES:
    d = masked[(rp, s)]
    u = (WEIGHTS["D"] * normalize(d["D"], "D") + WEIGHTS["V"] * normalize(d["V"], "V")
         + WEIGHTS["M"] * normalize(d["M"], "M"))
    fin = np.isfinite(d["D"]) & np.isfinite(d["V"]) & np.isfinite(d["M"])
    u = np.where(fin, u, np.nan); ufhi[(rp, s)] = u
    save(OUT_DIR / f"UFHI_{rp}_{s}.tif", u, d["prof"])
print("UFHI computed for", len(CASES), "cases.")

## 5. Jenks class breaks and classification

The continuous index is divided into five classes by the Jenks natural-breaks method (Jenks 1967). The breaks are computed once from a random sample of 150,000 values drawn from the pooled index of the 20 cases (random seed 0) and applied to every map. The area of each class follows from the cell counts, and the percentages are relative to the wetted area of each run, that is, the cells of that run with a simulated depth.

In [ ]:
allu = np.concatenate([valid(ufhi[k]) for k in CASES])
samp = np.random.default_rng(JENKS_SEED).choice(allu, size=min(JENKS_SAMPLE, allu.size), replace=False)
BREAKS = jenkspy.jenks_breaks(samp, n_classes=NCLASS); BREAKS[0], BREAKS[-1] = -np.inf, np.inf
pd.DataFrame({"Lower limit of class": HAZ_LABELS[1:], "UFHI": [float(b) for b in BREAKS[1:-1]]}
             ).to_csv(OUT_DIR / "jenks_breaks.csv", index=False)
print("Jenks breaks:", [round(float(b), 4) for b in BREAKS[1:-1]])

LAB = dict(enumerate(HAZ_LABELS, start=1))
rows = []
for rp, s in CASES:
    u = ufhi[(rp, s)]; cls = np.zeros(u.shape, dtype="int16"); fin = np.isfinite(u)
    cls[fin] = np.digitize(u[fin], BREAKS[1:-1], right=True) + 1
    tag = f"{rp}_{s}"
    save(OUT_DIR / f"UFHI_classified_{tag}.tif", cls.astype("float32"), masked[(rp, s)]["prof"], nodata=0)
    px = abs(masked[(rp, s)]["prof"]["transform"].a * masked[(rp, s)]["prof"]["transform"].e); tot = int(fin.sum())
    for ci in range(1, 6):
        n = int((cls == ci).sum())
        rows.append({"RP": rp, "Scenario": s, "Class": ci, "Hazard": LAB[ci],
                     "Cells": n, "Area_ha": round(n * px / 1e4, 3), "Pct": round(100 * n / tot, 2)})
area = pd.DataFrame(rows); area.to_csv(OUT_DIR / "hazard_class_areas.csv", index=False)

class_pct = (area.pivot(index=["RP", "Scenario"], columns="Hazard", values="Pct")
                 .reindex(index=CASES, columns=HAZ_LABELS))
class_pct.to_csv(OUT_DIR / "hazard_class_percentages.csv")
class_pct

## 6. Maximum depth, wetted area and high-hazard area

The High and Very High classes together are treated as high hazard.

In [ ]:
hi = area[area["Class"].isin([4, 5])].groupby(["RP", "Scenario"])["Pct"].sum().reset_index()
piv = hi.pivot(index="RP", columns="Scenario", values="Pct").reindex(RETURN_PERIODS)[SCENARIOS]
piv.to_csv(OUT_DIR / "high_hazard_pct.csv")

rows = []
for rp, s in CASES:
    sub = area[(area["RP"] == rp) & (area["Scenario"] == s)]
    rows.append({"RP": rp, "Scenario": s,
                 "Maximum depth (m)": round(float(np.nanmax(masked[(rp, s)]["D"])), 3),
                 "Wetted area (ha)": round(sub["Area_ha"].sum(), 1),
                 "High + Very High area (ha)": round(sub.loc[sub["Class"].isin([4, 5]), "Area_ha"].sum(), 1),
                 "High + Very High area (% of wetted area)": round(sub.loc[sub["Class"].isin([4, 5]), "Pct"].sum(), 2)})
summary = pd.DataFrame(rows); summary.to_csv(OUT_DIR / "flood_summary.csv", index=False)
print("High + Very High area (% of wetted area):")
display(piv)
summary

## 7. Class breaks expressed as flow conditions

Because the pooled minima of velocity and momentum are zero, the index of a cell with depth $D$ and velocity $V$ is linear in $D$, and the depth at which a class begins can be solved for a given velocity. The table gives the contributions of the three variables for an example cell and the depth at which the High and Very High classes begin in still water and in water moving at 0.2 m/s .

In [ ]:
(dmin, dmax), (vmin, vmax), (mmin, mmax) = BOUNDS["D"], BOUNDS["V"], BOUNDS["M"]

def contributions(D, V):
    return {"Depth": WEIGHTS["D"] * (D - dmin) / (dmax - dmin),
            "Velocity": WEIGHTS["V"] * (V - vmin) / (vmax - vmin),
            "Momentum": WEIGHTS["M"] * (D * V - mmin) / (mmax - mmin)}

ex = contributions(0.5, 0.2)
print("Cell with D = 0.5 m and V = 0.2 m/s:", {k: round(v, 3) for k, v in ex.items()},
      "| UFHI =", round(sum(ex.values()), 3))

def depth_at(ufhi_value, V):
    """Depth at which the index reaches ufhi_value for velocity V (valid for vmin = mmin = 0)."""
    slope = WEIGHTS["D"] / (dmax - dmin) + WEIGHTS["M"] * V / (mmax - mmin)
    rhs = ufhi_value + WEIGHTS["D"] * dmin / (dmax - dmin) - WEIGHTS["V"] * (V - vmin) / (vmax - vmin)
    return rhs / slope

rows = [{"Class": name, "UFHI break": round(float(b), 4),
         "Depth in still water (m)": round(depth_at(float(b), 0.0), 2),
         "Depth at 0.2 m/s (m)": round(depth_at(float(b), 0.2), 2)}
        for name, b in zip(HAZ_LABELS[1:], BREAKS[1:-1])]
thresholds = pd.DataFrame(rows); thresholds.to_csv(OUT_DIR / "class_break_depths.csv", index=False)
thresholds